# Olist Supply Chain & Delivery Performance Intelligence

## Python Analysis

### Objective
Use Python to perform deeper exploratory analysis and visualizations
to identify delivery, freight, and customer satisfaction risks.

The analysis complements the SQL phase by exploring relationships
between operational performance and customer satisfaction.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


Data Loading and dataset creation

In [18]:
customers = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_customers_dataset.csv")
order_items = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_order_items_dataset.csv")
reviews = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_order_reviews_dataset.csv")
products = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_products_dataset.csv")
orders = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_orders_dataset.csv")
sellers = pd.read_csv(r"F:\Data Science\My_Learning_Journey\olistst_supply_chain_decision_intelligence\Data\raw\olist_sellers_dataset.csv")

datasets = {
    "orders" : orders,
    "order_items" : order_items,
    "reviews" : reviews,
    "products" : products,
    "customers" : customers,
    "sellers" : sellers
}

for name, df in datasets.items(): print(f"{name}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
reviews: (99224, 7)
products: (32951, 9)
customers: (99441, 5)
sellers: (3095, 4)


Structure and data quality inspection

In [ ]:
for name, df in datasets.items(): 
    print(f"\n{'='*50}")
    print(name.upper())
    print(f"{'='*50}")
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

Problem found in datetime columns, fixint datetime columns

In [ ]:
order_date_time = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders[order_date_time] = orders[order_date_time].apply(pd.to_datetime)

# Convert shipping date in order_items
order_items['shipping_limit_date'] = pd.to_datetime(
    order_items['shipping_limit_date']
)
# Convert date columns in reviews

review_date_columns = [
    'review_creation_date',
    'review_answer_timestamp'
]

reviews[review_date_columns] = reviews[review_date_columns].apply(
    pd.to_datetime
)

# Verify
print(orders.dtypes)
print("\n")
print(order_items.dtypes)
print("\n")
print(reviews.dtypes)

Delivered orders dataset creation

In [32]:
orders['order_status'].value_counts()

delivered_orders= orders[
    orders['order_status']== 'delivered'
]

print(f'original orders: {orders.shape[0]}')
print(f'delivered_orders: {delivered_orders.shape[0]}')
print(f'percentage of delivered orders: {delivered_orders.shape[0]/orders.shape[0] * 100:.2f}%')

original orders: 99441
delivered_orders: 96478
percentage of delivered orders: 97.02%


In [35]:
delivery_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

print(
    delivered_orders[delivery_columns]
    .isnull()
    .sum()
)
print('=' *50)

analysis_orders = delivered_orders.dropna(
    subset=['order_delivered_customer_date']
).copy()

print(f"delivered orders before cleaning: {len(delivered_orders):,}")
print(f"order removed: {len(delivered_orders) - len(analysis_orders):,}")
print(f"Clean analysis orders: {len(analysis_orders):,}")


order_purchase_timestamp         0
order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64
delivered orders before cleaning: 96,478
order removed: 8
Clean analysis orders: 96,470


Finding delay orders

In [ ]:
analysis_orders['delivery_days'] = (
    analysis_orders['order_delivered_customer_date'] - analysis_orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

analysis_orders["estimated_delivery_days"] = (
    analysis_orders["order_estimated_delivery_date"] - analysis_orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

analysis_orders['delay_days'] = (
    analysis_orders['order_delivered_customer_date'] - analysis_orders['order_estimated_delivery_date']
).dt.total_seconds() / 86400

analysis_orders['delivery_status'] = np.where(
    analysis_orders['delay_days'] > 0, 
    'Late', 
    'On Time'
)
analysis_orders['high_delay']= np.where(
    analysis_orders['delay_days'] >= 8,
    'High_Delay',
    'Normal_Delay'
)

In [58]:
analysis_orders['delay_category'] = np.select(
    [
        analysis_orders['delay_days'] <= 0,
        (analysis_orders['delay_days'] > 0) & 
        (analysis_orders['delay_days'] <= 3),
        (analysis_orders['delay_days'] > 3) & 
        (analysis_orders['delay_days'] < 8),
        analysis_orders['delay_days'] >= 8
    ],
    [
        'On Time',
        'Low_Delay',
        'Medium_Delay',
        'High_Delay'
    ],
    default='unknown'
)

Validation of New Features

In [55]:
analysis_orders[
    [
    'delivery_days',
    'estimated_delivery_days',
    'delay_days'
    ]
].describe()

analysis_orders['delivery_status'].value_counts()
print()

analysis_orders['delay_category'].value_counts()

delay_category
On Time               88644
High_Delay (8+)        2862
Low_Delay (1-3)        2662
Medium_Delay (3-7)     2302
Name: count, dtype: int64

Delivery Features Summary

In [ ]:
print(
    analysis_orders[['delay_days', 'estimated_delivery_days', 'delivery_days']].describe()
)

print('\n Delivery Status:')
print(
    analysis_orders['delivery_status'].value_counts()
)

print('\nHigh Delay')
print(
    analysis_orders['delay_category'].value_counts()
)


         delay_days  estimated_delivery_days  delivery_days
count  96470.000000             96470.000000   96470.000000
mean     -11.178126                23.736343      12.558217
std       10.184354                 8.761052       9.546156
min     -146.016123                 2.008009       0.533414
25%      -16.244065                18.329905       6.766204
50%      -11.948102                23.230880      10.217477
75%       -6.389815                28.407795      15.720182
max      188.975081               155.135463     209.628611

 Delivery Status:
delivery_status
On Time    88644
Late        7826
Name: count, dtype: int64

High Delay
delay_category
On Time               88644
High_Delay (8+)        2862
Low_Delay (1-3)        2662
Medium_Delay (3-7)     2302
Name: count, dtype: int64


Delivery Feature Engineering Completed

Review Table Analysis

In [59]:
print("Total review rowa", len(reviews))
print("Unique order_id with reviews", reviews['order_id'].unique())
print("Duplicate order_id", reviews['order_id'].duplicated().sum())

Total review rowa 99224
Unique order_id with reviews ['73fc7af87114b39712e6da79b0a377eb' 'a548910a1c6147796b98fdf73dbeba33'
 'f9e4b658b201a9f2ecdecbb34bed034b' ... '55d4004744368f5571d1f590031933e4'
 '7725825d039fc1f0ceb7635e3f7d9206' '90531360ecb1eec2a1fbb265a0db0508']
Duplicate order_id 551


In [ ]:
multiple_reviews = reviews[
    reviews.duplicated('order_id', keep=False)
].sort_values('order_id')

multiple_reviews[
    ['order_id', 'review_id', 'review_score']
].head(20)



,order_id,review_id,review_score
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5


In [61]:
print("Total review rows:", len(reviews))
print("Unique orders with reviews:", reviews['order_id'].nunique())
print("Orders with multiple reviews:", multiple_reviews['order_id'].nunique())
print("Rows belonging to multiple-review orders:", len(multiple_reviews))

Total review rows: 99224
Unique orders with reviews: 98673
Orders with multiple reviews: 547
Rows belonging to multiple-review orders: 1098


In [ ]:
multiple_reviews[
    ['order_id', 'review_id', 'review_score',
     'review_creation_date', 'review_answer_timestamp']
].head(20)


Handling Multiple reviews with one order id to keep one order one review grain

In [64]:
review_per_order = (
    reviews.groupby('order_id').agg(
        review_score=('review_score', 'mean'),
        review_count=('review_id', 'count'),
        first_review_date=('review_creation_date', 'min'),
        last_review_date=('review_creation_date', 'max')
    ).reset_index()
)

print("Total rows:", len(review_per_order))
print("Unique order IDs:", review_per_order['order_id'].nunique())
print("Duplicate order IDs:", review_per_order['order_id'].duplicated().sum())

Total rows: 98673
Unique order IDs: 98673
Duplicate order IDs: 0


Merging order and review dataset 

In [66]:
analysis_orders = analysis_orders.merge(
    review_per_order,
    on= 'order_id',
    how='left',
    validate= 'one_to_one'
)

In [68]:
print(analysis_orders.shape)
print("\n Missing reviw score:")
print(analysis_orders['review_score'].isna().sum())
print("\n Review score distribution")
print(analysis_orders['review_score'].value_counts().sum())


(96470, 17)

 Missing reviw score:
646

 Review score distribution
95824


Merging customers

In [69]:
analysis_orders = analysis_orders.merge(
    customers[
        ['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']
    ],
    on='customer_id',
    how='left',
    validate='one_to_one'
)

In [72]:
print(analysis_orders.shape)
print(
    analysis_orders[
        ['customer_unique_id', 'customer_city', 'customer_state']
    ].isna().sum()
)

(96470, 20)
customer_unique_id    0
customer_city         0
customer_state        0
dtype: int64


Order items dataset analysis

In [73]:
order_items_per_order = (
    order_items.groupby('order_id').agg(
        total_items = ('order_item_id', 'count'),
        total_product_value = ('price', 'sum'),
        total_freight_value = ('freight_value', 'sum'),
        unique_products = ('product_id', 'nunique'),
        unique_sellers = ('seller_id', 'nunique')
    ).reset_index()
)

order_items_per_order.head

<bound method NDFrame.head of                                order_id  total_items  total_product_value  \
0      00010242fe8c5a6d1ba2dd792cb16214            1                58.90   
1      00018f77f2f0320c557190d7a144bdd3            1               239.90   
2      000229ec398224ef6ca0657da4fc703e            1               199.00   
3      00024acbcdf0a6daa1e931b038114c75            1                12.99   
4      00042b26cf59d7ce69dfabb4e55b4fd9            1               199.90   
...                                 ...          ...                  ...   
98661  fffc94f6ce00a00581880bf54a75a037            1               299.99   
98662  fffcd46ef2263f404302a634eb57f7eb            1               350.00   
98663  fffce4705a9662cd70adb13d4a31832d            1                99.90   
98664  fffe18544ffabc95dfada21779c9644f            1                55.99   
98665  fffe41c64501cc87c801fd61db3f6244            1                43.00   

       total_freight_value  unique_products  

In [ ]:
analysis_orders = analysis_orders.merge(
    order_items_per_order,
    on= 'order_id',
    how= 'left',
    validate= 'one_to_one'
)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delay_days,delivery_status,delay_category,review_score,review_count,first_review_date,last_review_date,customer_unique_id,customer_city,customer_state,total_items,total_product_value,total_freight_value,unique_products,unique_sellers
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,On Time,On Time,4.0,1.0,2017-10-11,2017-10-11,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1,29.99,8.72,1,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,On Time,On Time,4.0,1.0,2018-08-08,2018-08-08,af07308b275d755c9edb36a90c618231,barreiras,BA,1,118.70,22.76,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,On Time,On Time,5.0,1.0,2018-08-18,2018-08-18,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,1,159.90,19.22,1,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,-12.980069,On Time,On Time,5.0,1.0,2017-12-03,2017-12-03,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1,45.00,27.20,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,-9.238171,On Time,On Time,5.0,1.0,2018-02-17,2018-02-17,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,1,19.90,8.72,1,1


In [75]:
print(analysis_orders.shape)

print(
    analysis_orders[
        [
            'total_items',
            'total_product_value',
            'total_freight_value',
            'unique_products',
            'unique_sellers'
        ]
    ].isna().sum()
)

(96470, 25)
total_items            0
total_product_value    0
total_freight_value    0
unique_products        0
unique_sellers         0
dtype: int64


Products and category dataset merging to analysis_orders dataset

In [76]:
order_product_detail = order_items.merge(
    products[['product_id', 'product_category_name']],
    on= 'product_id',
    how= 'left'
)

In [79]:
order_categories = (
    order_product_detail.groupby('order_id').aggregate(
    unique_categories = ('product_category_name', 'nunique')
).reset_index()
)

print(order_categories.shape)
print('Duplicate orders', order_categories['order_id'].duplicated().sum())
print(order_categories['unique_categories'].value_counts().sort_index())

(98666, 2)
Duplicate orders 0
unique_categories
0     1389
1    96550
2      712
3       15
Name: count, dtype: int64


In [80]:
analysis_orders = analysis_orders.merge(
    order_categories,
    on='order_id',
    how= 'left',
    validate= 'one_to_one'
)

print(analysis_orders.shape)
print(analysis_orders['unique_categories'].isna().sum())

(96470, 26)
0


Seller Level Dataset Creation

In [84]:
seller_order_details = analysis_orders.merge(
    order_items[
        ['order_id', 'seller_id']],
    on= 'order_id',
    how= 'left'
)

print(seller_order_details.shape)
print(seller_order_details[['order_id', 'seller_id']].isna().sum())

(110189, 27)
order_id     0
seller_id    0
dtype: int64


In [86]:
seller_order_details.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delay_days,delivery_status,delay_category,review_score,review_count,first_review_date,last_review_date,customer_unique_id,customer_city,customer_state,total_items,total_product_value,total_freight_value,unique_products,unique_sellers,unique_categories,seller_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,15.544063,-7.107488,On Time,On Time,4.0,1.0,2017-10-11,2017-10-11,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1,29.99,8.72,1,1,1,3504c0cb71d7fa48d967e0e4c94d59d9
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,19.137766,-5.355729,On Time,On Time,4.0,1.0,2018-08-08,2018-08-08,af07308b275d755c9edb36a90c618231,barreiras,BA,1,118.70,22.76,1,1,1,289cdb325fb7e7f891c38608bf9e0962
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,26.639711,-17.245498,On Time,On Time,5.0,1.0,2018-08-18,2018-08-18,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,1,159.90,19.22,1,1,1,4869f7a5dfa277a7dca6462dcf3b52b2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,26.188819,-12.980069,On Time,On Time,5.0,1.0,2017-12-03,2017-12-03,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1,45.00,27.20,1,1,1,66922902710d126a0e7d26b0e3805106
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,12.112049,-9.238171,On Time,On Time,5.0,1.0,2018-02-17,2018-02-17,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,1,19.90,8.72,1,1,1,2c9e548be18521d1c43cde1c582c6de8


In [91]:
seller_order_details.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delay_days',
 'delivery_status',
 'delay_category',
 'review_score',
 'review_count',
 'first_review_date',
 'last_review_date',
 'customer_unique_id',
 'customer_city',
 'customer_state',
 'total_items',
 'total_product_value',
 'total_freight_value',
 'unique_products',
 'unique_sellers',
 'unique_categories',
 'seller_id']

In [111]:
seller_order_details['is_dissatisfied'] = seller_order_details['review_score'].isin([1,2])
seller_order_details['is_late'] = seller_order_details['delivery_status'].eq('Late')
seller_order_details['is_high_delay'] = seller_order_details['delay_category'].eq('High_Delay')

In [112]:
seller_analysis = (
    seller_order_details
    .groupby('seller_id')
    .agg(
        total_orders=('order_id', 'nunique'),
        average_price=('total_product_value', 'mean'),
        average_freight=('total_freight_value', 'mean'),
        average_review=('review_score', 'mean'),
        dissatisfied_customers=('is_dissatisfied', 'sum'),
        late_orders=('is_late', 'sum'),
        high_delay_orders=('is_high_delay', 'sum')
    )
    .reset_index()
)

In [113]:
seller_analysis['dissatisfied_rate'] = (
    seller_analysis['dissatisfied_customers'] / seller_analysis['total_orders'] * 100
)

seller_analysis['high_delay_rate']  = (
    seller_analysis['high_delay_orders'] / seller_analysis['total_orders'] * 100 
)

In [114]:
seller_analysis.shape
print(seller_analysis.head(5))

                          seller_id  total_orders  average_price  \
0  0015a82c2db000af6aaaf3ae2ecb0532             3     895.000000   
1  001cca7ae9ae17fb1caed9dfb1094831           195     151.132094   
2  002100f778ceb8431b7a1020ff7ab48f            50      27.401852   
3  003554e2dce176b5555353e4f3555ac8             1     120.000000   
4  004c9cd9d87a3c30c522c48c4fc07416           156     140.476131   

   average_freight  average_review  dissatisfied_customers  late_orders  \
0        21.020000        3.666667                       1            0   
1        52.107094        3.965368                      40           13   
2        17.276667        4.037037                       7            9   
3        19.380000        5.000000                       0            0   
4        24.050238        4.156627                      20           13   

   high_delay_orders  dissatisfied_rate  high_delay_rate  
0                  0          33.333333         0.000000  
1                  5  

In [115]:
seller_analysis = seller_analysis[
    seller_analysis['total_orders'] >= 50
]

In [116]:
print(seller_analysis.shape)
print(seller_analysis['total_orders'].min())

(425, 10)
50


In [117]:
high_delay_average = seller_analysis['high_delay_rate']. mean()
average_dissatisfied = seller_analysis['dissatisfied_rate'].mean()
average_reviews = seller_analysis['average_review'].mean()

print(high_delay_average)
print(average_dissatisfied)
print(average_reviews)

3.215461180619764
16.411398983810898
4.100493627698654


In [120]:
conditions = [
    (seller_analysis['high_delay_rate'] > 3.2) &
        (seller_analysis['dissatisfied_rate'] >16.4),
    (seller_analysis['high_delay_rate'] <= 3.2) &
        (seller_analysis['dissatisfied_rate'] >16.4),
    (seller_analysis['high_delay_rate'] > 3.2) &
        (seller_analysis['dissatisfied_rate'] <=16.4),
    (seller_analysis['high_delay_rate'] <= 3.2) &
        (seller_analysis['dissatisfied_rate'] <=16.4)       
]
choices =[
       'Critical_Seller',
       'Customer_Product_Risk',
       'Delivery_Risk',
       'Strong_Performer'
]

seller_analysis['seller_segmentation'] = np.select(
    conditions,
    choices,
    default='unknown'
)

In [125]:
seller_analysis['seller_segmentation'].value_counts(normalize=True).mul(100) .__round__(1)

seller_segmentation
Strong_Performer         40.9
Critical_Seller          22.6
Customer_Product_Risk    18.4
Delivery_Risk            18.1
Name: proportion, dtype: float64

Category Dataset Creation


In [129]:
analysis_orders.columns.tolist()
order_categories.head()
print(order_categories.shape)
print(order_categories.columns)
print(order_categories['order_id'].duplicated().sum())

(98666, 2)
Index(['order_id', 'unique_categories'], dtype='object')
0


In [130]:
order_category_details = (
    order_items[
        ['order_id', 'product_id']
    ]
    .merge(
        products[
            ['product_id', 'product_category_name']
        ],
        on='product_id',
        how='left',
        validate='many_to_one'
    )
    [['order_id', 'product_category_name']]
    .drop_duplicates()
)

In [131]:
print(order_category_details.shape)
print(order_category_details.head())
print(order_category_details['order_id'].duplicated().sum())

(99470, 2)
                           order_id product_category_name
0  00010242fe8c5a6d1ba2dd792cb16214            cool_stuff
1  00018f77f2f0320c557190d7a144bdd3              pet_shop
2  000229ec398224ef6ca0657da4fc703e      moveis_decoracao
3  00024acbcdf0a6daa1e931b038114c75            perfumaria
4  00042b26cf59d7ce69dfabb4e55b4fd9    ferramentas_jardim
804


In [132]:
category_order_details = order_category_details.merge(
    analysis_orders[
        [
            'order_id',
            'review_score',
            'delay_days',
            'delivery_status',
            'delay_category'
        ]
    ],
    on='order_id',
    how='inner',
    validate='many_to_one'
)

In [134]:
print(category_order_details.shape)

print(category_order_details.isna().sum())

print(category_order_details.head())

(97268, 6)
order_id                    0
product_category_name    1392
review_score              652
delay_days                  0
delivery_status             0
delay_category              0
dtype: int64
                           order_id product_category_name  review_score  \
0  00010242fe8c5a6d1ba2dd792cb16214            cool_stuff           5.0   
1  00018f77f2f0320c557190d7a144bdd3              pet_shop           4.0   
2  000229ec398224ef6ca0657da4fc703e      moveis_decoracao           5.0   
3  00024acbcdf0a6daa1e931b038114c75            perfumaria           4.0   
4  00042b26cf59d7ce69dfabb4e55b4fd9    ferramentas_jardim           5.0   

   delay_days delivery_status delay_category  
0   -8.011250         On Time        On Time  
1   -2.330278         On Time        On Time  
2  -13.444954         On Time        On Time  
3   -5.435660         On Time        On Time  
4  -15.303808         On Time        On Time  


In [135]:
category_order_details_clean = category_order_details.dropna(
    subset=['product_category_name', 'review_score']
)

In [139]:
category_order_details_clean['is_dissatisfied'] =(category_order_details_clean['review_score'] <=2 )

C:\Users\MBS TRADERS\AppData\Local\Temp\ipykernel_11284\3524799440.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_order_details_clean['is_dissatisfied'] =(category_order_details_clean['review_score'] <=2 )


In [140]:
category_analysis = (
    category_order_details_clean.groupby('product_category_name')
    .agg(
        total_orders = ('order_id','count'),
        average_reviews=('review_score', 'mean'),
        dissatisfied_customers = ('is_dissatisfied', 'sum')
    ).reset_index()
)

In [144]:
category_analysis['dissatisfied_rate'] =(
    category_analysis['dissatisfied_customers'] / category_analysis['total_orders']* 100
)

In [146]:
category_analysis = category_analysis[category_analysis['total_orders'] >= 50]

In [148]:
category_analysis.shape
category_analysis.head()
print(category_analysis['total_orders'].min())

50


Benchmarks calculations for refrence

In [149]:
average_dissatisfied = category_analysis['dissatisfied_rate'].mean()
average_review = category_analysis['average_reviews'].mean()

print(average_reviews)
print(average_dissatisfied)

4.100493627698654
12.591024414133832


In [159]:
conditions = [
        (category_analysis['dissatisfied_rate'] > 12.59)&
            (category_analysis['average_reviews'] < 4.10),
        (category_analysis['dissatisfied_rate'] > 12.59)&
            (category_analysis['average_reviews'] >= 4.10),
        (category_analysis['dissatisfied_rate'] <= 12.59)&
            (category_analysis['average_reviews'] < 4.10),
        (category_analysis['dissatisfied_rate'] <= 12.59)&
            (category_analysis['average_reviews'] >= 4.10)
    ]
    
choices=[
        ('Critical_Category'),
        ('Dissatisfaction_Risk'),
        ('Low_Satisfaction'),
        ('Strong_Category')
    ]

category_analysis['category_segmentation'] = np.select(
    conditions,
    choices,
    default= 'Unknown'
)

In [160]:
category_analysis['category_segmentation'].value_counts()

category_segmentation
Strong_Category         35
Critical_Category       14
Dissatisfaction_Risk     8
Low_Satisfaction         1
Name: count, dtype: int64

Critical Categories ranking by number of orders

In [161]:
critical_categories= category_analysis[category_analysis['category_segmentation'] == 'Critical_Category'].copy()

critical_categories['business_priority_rank'] = (
    critical_categories['dissatisfied_customers'].rank(method='dense', ascending=False)
)

In [162]:
critical_categories.sort_values(
    'business_priority_rank'
).head(10)

,product_category_name,total_orders,average_reviews,dissatisfied_customers,dissatisfied_rate,category_segmentation,business_priority_rank
13,cama_mesa_banho,9177,3.999292,1459,15.898442,Critical_Category,1.0
54,moveis_decoracao,6260,4.057987,958,15.303514,Critical_Category,2.0
44,informatica_acessorios,6498,4.081717,941,14.481379,Critical_Category,3.0
70,telefonia,4069,4.051855,575,14.131236,Critical_Category,4.0
55,moveis_escritorio,1244,3.642685,273,21.945338,Critical_Category,5.0
16,casa_construcao,481,3.985447,83,17.255717,Critical_Category,6.0
7,audio,345,3.839130,75,21.739130,Critical_Category,7.0
14,casa_conforto,390,3.887179,73,18.717949,Critical_Category,8.0
57,moveis_sala,409,4.070905,58,14.180929,Critical_Category,9.0
71,telefonia_fixa,209,3.966507,35,16.746411,Critical_Category,10.0


In [164]:
analysis_orders['freight_price_ratio'] = (
    analysis_orders['total_freight_value']
    /analysis_orders['total_product_value']
) * 100

In [165]:
analysis_orders[
    ['total_product_value', 'total_freight_value', 'freight_price_ratio']
].describe()

,total_product_value,total_freight_value,freight_price_ratio
count,96470.000000,96470.000000,96470.000000
mean,137.040001,22.785798,30.838580
std,209.052608,21.559959,31.161754
min,0.850000,0.000000,0.000000
25%,45.900000,13.850000,13.201681
50%,86.500000,17.170000,22.437396
75%,149.900000,24.020000,38.056680
max,13440.000000,1794.960000,2144.705882


In [167]:
analysis_orders['is_dissatisfied'] = (
    analysis_orders['review_score'] <= 2
)

In [168]:
freight_satisfaction = (
    analysis_orders
    .dropna(subset=['review_score'])
    .groupby('is_dissatisfied')
    .agg(
        total_orders=('order_id', 'count'),
        avg_freight_ratio=('freight_price_ratio', 'mean'),
        median_freight_ratio=('freight_price_ratio', 'median'),
        avg_freight=('total_freight_value', 'mean')
    )
    .reset_index()
)

freight_satisfaction

,is_dissatisfied,total_orders,avg_freight_ratio,median_freight_ratio,avg_freight
0,False,83588,30.668414,22.419048,22.036358
1,True,12236,32.049358,23.306464,27.725960


In [163]:
critical_categories.to_csv(
    'critical_categories.csv',
    index=False
)

In [169]:
analysis_orders.to_csv('analysis_orders.csv', index=False)

seller_analysis.to_csv('seller_analysis.csv', index=False)

category_analysis.to_csv('category_analysis.csv', index=False)

freight_satisfaction.to_csv('freight_satisfaction.csv', index=False)